In [1]:
import numpy as np
import os 
from PIL import Image

In [2]:
# Activation functions
def relu(x):
    return np.maximum(0, x)
    
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return (1 / (1 + np.exp(-x)))

input_nodes = 784  #28x28 pxl
hidden_nodes = 100
output_nodes = 3  # pig, pug, plopp


# Weights. Row - Source, Col - Destination ie n1 - w11, w12, w13 
w_inp_hid = np.random.randn(input_nodes, hidden_nodes) * 0.01    #np.sqrt(2 / input_nodes)
w_hid_out = np.random.randn(hidden_nodes, output_nodes) * 0.01   # np.sqrt(2 / hidden_nodes)


# Biases - one per node (shown as per layer)
b1 = np.zeros((1, hidden_nodes))
b2 = np.zeros((1, output_nodes))

print(f"Weights range: {w_inp_hid.min():.4f} to {w_inp_hid.max():.4f}")

Weights range: -0.0473 to 0.0400


In [3]:
# images

def load_image(folder_path):
    images = []
    labels = []

    # map the folder names to the node targets
    categories = {
                'pig': [1, 0, 0],
                'pug': [0, 1, 0],
                'plopp': [0, 0, 1]}

    for category, target in categories.items(): # items allow to go for both key, value
        path = os.path.join(folder_path, category)
        for file in os.listdir(path):
            if file.endswith(('.jpg', 'jpeg', 'png')): 
                # load them: 
                img = Image.open(os.path.join(path, file)).convert('L') #make b/w
                img = img.resize((28, 28))  # 28x28 pixels

                # flatten & make array
                img_data = np.array(img).flatten() / 255.0   # bc 255 is the highest L.value

                images.append(img_data)
                labels.append(target)

    return np.array(images), np.array(labels)


# Loading Images
X, y = load_image('ppp')

print(f"X shape: {X.shape}")  # Should be (15, 784)
print(f"y shape: {y.shape}")  # Should be (15, 3)

X shape: (15, 784)
y shape: (15, 3)


In [4]:
# Backpropagation set up

# sigmoid derivative - output x (1-output)
# ReLu derivative - 1 if input was positive, 0 if it was negative 

def sig_deriv(output):
    return output * (1 - output)

def relu_deriv(x):
    return (x > 0).astype(float)

In [5]:
# TRAINING LOOP

learning_rate = 0.01
epochs = 500

for epoch in range(epochs):
    total_loss = 0 

    for i in range(len(X)):

    
        # --- 1. Forward pass ---
        input_layer = X[i:i+1]   # shape (1, 784) - 1D array, 784 'numbers'
    
        # Hidden Layer
        hid_raw = input_layer @ w_inp_hid + b1
        hid_act = relu(hid_raw)  # bc relu is a function relu(x)
    
        # Output Layer
        out_raw = hid_act @ w_hid_out + b2
        out_act = sigmoid(out_raw) # bc sigmoid is a func sigmoid(output)
        #out_act = out_raw
    
    
        # --- 2. Error Calculation
        # Target - Prediciton(Output)
    
        error = y[i:i+1] - out_act
        total_loss += np.sum(error**2)  # choose to look at the error using exp
    
    
        # --- 3. Backpropagation
    
        # Error signal for output layer (sigmoid)
        output_delta = error * sig_deriv(out_act)    
        #output_delta = error
        
        # Error from nodes in Hidden Layer => use .T
        hidden_error = output_delta @ w_hid_out.T
        
        # Error signal for hidden layer (ReLu) 
        hidden_delta = hidden_error * relu_deriv(hid_raw)

        # Inside the training loop, after backprop:
        if epoch % 50 == 0 and i == 0:
            print(f"Max Gradient: {np.max(np.abs(hidden_delta))}")
            print(f"Max Weight Change: {np.max(np.abs((input_layer.T @ hidden_delta) * learning_rate))}")
            
    
        # --- 4. Update Weights 
        # New Weight = Old Weight + (Input x Delta x Learning Rate)
        w_hid_out += (hid_act.T @ output_delta) * learning_rate
        w_inp_hid += (input_layer.T @ hidden_delta) * learning_rate
    
        # Update Biases
        b2 += output_delta * learning_rate
        b1 += hidden_delta * learning_rate

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Max Gradient: 0.005506084303965964
Max Weight Change: 5.0094570922357e-05
Epoch 0, Loss: 11.2647
Max Gradient: 0.011946130615417539
Max Weight Change: 0.00010868636481477917
Epoch 50, Loss: 9.5389
Max Gradient: 0.02372231669562785
Max Weight Change: 0.00021582656758375142
Epoch 100, Loss: 6.5251
Max Gradient: 0.01795137370075696
Max Weight Change: 0.00016332230190492603
Epoch 150, Loss: 3.1339
Max Gradient: 0.009813874168413676
Max Weight Change: 8.928701204203814e-05
Epoch 200, Loss: 1.6127
Max Gradient: 0.00528697725616421
Max Weight Change: 4.81011264090234e-05
Epoch 250, Loss: 0.8005
Max Gradient: 0.002990484821618649
Max Weight Change: 2.720754818100104e-05
Epoch 300, Loss: 0.3933
Max Gradient: 0.001739960205662813
Max Weight Change: 1.583022618485383e-05
Epoch 350, Loss: 0.2326
Max Gradient: 0.0013825677935067212
Max Weight Change: 1.2578656003668991e-05
Epoch 400, Loss: 0.1582
Max Gradient: 0.0010688199451894058
Max Weight Change: 9.724165775840868e-06
Epoch 450, Loss: 0.1178


In [6]:
# Saving the weights

save_dir ='saved_weights'

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

np.save(os.path.join(save_dir, 'weights_input_hidden.npy'), w_inp_hid)
np.save(os.path.join(save_dir, 'weights_hidden_output.npy'), w_hid_out)
np.save(os.path.join(save_dir, 'b1.npy'), b1)
np.save(os.path.join(save_dir, 'b2.npy'), b2)

print(f"Brain saved successfully in {save_dir}/")

Brain saved successfully in saved_weights/
